# R2 Bucket Inspector

Using boto3, authenticate and reach the `cytemaps` R2 bucket and inspect its contents.

In [ ]:
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv
from shared.repo import REPO_ROOT

load_dotenv(dotenv_path=REPO_ROOT / ".env")

BUCKET = os.environ["BUCKET"]
ENDPOINT_URL = os.environ["ENDPOINT_URL"]

# Test file name
KEY = "test.txt"

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

print(f"Endpoint : {ENDPOINT_URL}")
print(f"Bucket   : {BUCKET}")

## List all objects, dump in cache

In [ ]:
CACHE_DIR = REPO_ROOT / "output/r2"
os.makedirs(CACHE_DIR, exist_ok=True)
with open(CACHE_DIR / "r2_files_cache.txt", "w") as f:
    f.write("")

paginator = client.get_paginator("list_objects_v2")
objects = []
for page in paginator.paginate(Bucket=BUCKET):
    objects.extend(page.get("Contents") or [])

if not objects:
    print("Bucket is empty.")
else:
    print(f"{len(objects)} object(s):")
    for obj in objects:
        size_mb = obj["Size"] / 1024 / 1024
        uploaded = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)  {uploaded}")
        

        with open(CACHE_DIR / "r2_files_cache.txt", "a") as f:
            f.write(f"  {obj['Key']}  ({size_mb:.1f} MB)  {uploaded}\n")

## Show unique prefixes

In [ ]:
# List all unique file prefixes (i.e., parts before the first '/')
unique_prefixes = set()
for obj in objects:
    key = obj["Key"]
    prefix = key.split("/", 1)[0] if "/" in key else key
    unique_prefixes.add(prefix)
if not unique_prefixes:
    print("No prefixes found.")
else:
    print(f"Unique prefixes ({len(unique_prefixes)}):")
    for p in sorted(unique_prefixes):
        print(f"- {p}")

## Filter by prefix

In [ ]:
PREFIX = "designated_clustering_2/"
# PREFIX = "cytetype_pipeline_20260522_175813/"

paginator = client.get_paginator("list_objects_v2")
filtered = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    filtered.extend(page.get("Contents") or [])

if not filtered:
    print(f"No objects found under '{PREFIX}'.")
else:
    print(f"{len(filtered)} object(s) under '{PREFIX}':")
    for obj in filtered:
        size_mb = obj["Size"] / 1024 / 1024
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")

# Modifying files on R2
## Delete files with a given prefix

In [ ]:
DELETE_PREFIX = "deleteme3/"

paginator = client.get_paginator("list_objects_v2")
to_delete = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=DELETE_PREFIX):
    to_delete.extend(page.get("Contents") or [])

if not to_delete:
    print(f"No objects found under '{DELETE_PREFIX}'. Nothing to delete.")
else:
    print(f"Found {len(to_delete)} object(s) under '{DELETE_PREFIX}':")
    total_mb = 0.0
    for obj in to_delete:
        size_mb = obj["Size"] / 1024 / 1024
        total_mb += size_mb
        print(f"  {obj['Key']}  ({size_mb:.1f} MB)")
    print(f"\nTotal: {total_mb:.1f} MB")

    confirm = input(f"\nType 'yes' to delete all {len(to_delete)} objects: ")
    if confirm.strip().lower() != "yes":
        print("Aborted.")
    else:
        # S3 delete_objects accepts up to 1000 keys per call
        for i in range(0, len(to_delete), 1000):
            batch = to_delete[i : i + 1000]
            resp = client.delete_objects(
                Bucket=BUCKET,
                Delete={"Objects": [{"Key": obj["Key"]} for obj in batch], "Quiet": True},
            )
            errors = resp.get("Errors", [])
            if errors:
                for err in errors:
                    print(f"  ERROR: {err['Key']} - {err['Message']}")
        print(f"Deleted {len(to_delete)} object(s) under '{DELETE_PREFIX}'.")

In [ ]:
from pathlib import Path

p = REPO_ROOT / "scripts" / "cluster_validation" / "cell_type_metrics.py"
p.parent

## Upload a test file

In [ ]:
import io

body = b"hello from r2_test.ipynb"

client.put_object(Bucket=BUCKET, Key=KEY, Body=body)
print(f"Uploaded '{KEY}' ({len(body)} bytes) to r2://{BUCKET}/{KEY}")

## Read the test file back

In [ ]:
response = client.get_object(Bucket=BUCKET, Key=KEY)
content = response["Body"].read().decode()
print(f"Contents of '{KEY}': {content!r}")

## Download the test file to repo root

In [ ]:
from shared.repo import REPO_ROOT

KEY = "arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX22996378.h5ad"

dest = REPO_ROOT / "tmp" / "SRX22996378_from_r2.h5ad"
client.download_file(BUCKET, KEY, str(dest))
print(f"Downloaded '{KEY}' -> {dest}")
# print(f"Contents: {dest.read_text()!r}")